#**This Notebook is for Section II: Data Preprocessing.**

You can run this entire notebook to carry out the datapreprocessing steps.

In [ ]:
# Mount folder onto Notebook
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [ ]:
# Change directory
import os
import sys
path = '/content/drive/My Drive/cs5246-code/' #add shortcut to
os.chdir(path)
intermediate_csv_path = path + 'intermediate_csvs/'
original_csv_path = path + 'data/original_dataset/'
auxiliary_data_path = path + 'data/auxiliary_data/'
src_path = path + 'src/'
sys.path.append(src_path)

In [ ]:
# Load necessary libraries
# TODO: Remove unused libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import seaborn as sns
import re
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math
import geopandas as gpd
from shapely.geometry import Point
np.set_printoptions(precision=3)
import dev.data_prep as dataprep

# Section II-A: Load Data

In [ ]:
train_dev_df = pd.read_csv(original_csv_path + 'train.csv')
test_df = pd.read_csv(original_csv_path + 'test.csv')

# Section II-B: Data Cleaning and Feature Transformation

In [ ]:
clean_train_dev_df = dataprep.clean_data(train_dev_df)
clean_test_df = dataprep.clean_data(test_df)

In [ ]:
dataprep.df_basic_info(clean_train_dev_df, table_name="Train/Dev Set")
dataprep.df_basic_info(clean_test_df, table_name="Test Set")


=== Train/Dev Set summary ===
╒═══════════════════════════╤════════╕
│ Number of records         │ 162691 │
├───────────────────────────┼────────┤
│ Number of features        │     12 │
├───────────────────────────┼────────┤
│ Number of NaN rows        │      0 │
├───────────────────────────┼────────┤
│ Number of duplicated rows │    452 │
╘═══════════════════════════╧════════╛

=== Test Set summary ===
╒═══════════════════════════╤═══════╕
│ Number of records         │ 50000 │
├───────────────────────────┼───────┤
│ Number of features        │    11 │
├───────────────────────────┼───────┤
│ Number of NaN rows        │     0 │
├───────────────────────────┼───────┤
│ Number of duplicated rows │   506 │
╘═══════════════════════════╧═══════╛


`# Section II-C: Duplicate Removal and Data Partitioning `

In [ ]:
nodup_train_dev_df = clean_train_dev_df.drop_duplicates(keep='first').reset_index(drop=True)
train_df, dev_df = train_test_split(
    nodup_train_dev_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

test_df = clean_test_df

In [ ]:
dataprep.df_basic_info(nodup_train_dev_df, table_name="Train/Dev Set After Duplicate Removal")
dataprep.df_basic_info(train_df, table_name="Train Set After Split")
dataprep.df_basic_info(dev_df, table_name="Dev Set After Split")


=== Train/Dev Set After Duplicate Removal summary ===
╒═══════════════════════════╤════════╕
│ Number of records         │ 162464 │
├───────────────────────────┼────────┤
│ Number of features        │     12 │
├───────────────────────────┼────────┤
│ Number of NaN rows        │      0 │
├───────────────────────────┼────────┤
│ Number of duplicated rows │      0 │
╘═══════════════════════════╧════════╛

=== Train Set After Split summary ===
╒═══════════════════════════╤════════╕
│ Number of records         │ 129971 │
├───────────────────────────┼────────┤
│ Number of features        │     12 │
├───────────────────────────┼────────┤
│ Number of NaN rows        │      0 │
├───────────────────────────┼────────┤
│ Number of duplicated rows │      0 │
╘═══════════════════════════╧════════╛

=== Dev Set After Split summary ===
╒═══════════════════════════╤═══════╕
│ Number of records         │ 32493 │
├───────────────────────────┼───────┤
│ Number of features        │    12 │
├──────────────

In [ ]:
#nodup_train_dev_df.to_csv(intermediate_csv_path + 'no_dup_train.csv', index=False)
#dev_df.to_csv(intermediate_csv_path + 'no_dup_val.csv', index=False)

# Section II-D: Auxiliary Data Processing and Feature Enrichment

In [ ]:
# Load main HDB resale dataset
hdb_df = pd.read_csv(auxiliary_data_path + 'sg-hdb-block-details.csv')
mall_df = pd.read_csv(auxiliary_data_path + 'sg-shopping-malls.csv')
hawker_df = pd.read_csv(auxiliary_data_path + 'sg-gov-hawkers.csv')
pri_sch_df = pd.read_csv(auxiliary_data_path + 'sg-primary-schools.csv')
sec_sch_df = pd.read_csv(auxiliary_data_path + 'sg-secondary-schools.csv')
walking_dist_df = pd.read_csv(auxiliary_data_path + 'hdb-block-walking-distance.csv')
mrt_df = pd.read_csv(auxiliary_data_path + 'sg-mrt-stations.csv')

In [ ]:
# Process auxiliary data
mall_df = dataprep.process_mall_df(mall_df)
hawker_df = dataprep.process_hawker_df(hawker_df)
pri_sch_df = dataprep.process_sch_df(pri_sch_df)
sec_sch_df = dataprep.process_sch_df(sec_sch_df)
walking_dist_df = dataprep.process_walk_dist_df(walking_dist_df)
mrt_df_dict = dataprep.process_mrt_df(mrt_df)

In [ ]:
# Enrich HDB data
processed_hdb_df = dataprep.process_hdb_df(
    hdb_df=hdb_df,
    mrt_df_dict=mrt_df_dict,
    mall_df=mall_df,
    hawker_df=hawker_df,
    pri_sch_df=pri_sch_df,
    sec_sch_df=sec_sch_df,
    walking_dist_df=walking_dist_df
)

In [ ]:
# Integrate auxiliary data into train/dev/test
integrated_train_df = dataprep.integrate_aux_data(train_df, processed_hdb_df)
integrated_dev_df = dataprep.integrate_aux_data(dev_df, processed_hdb_df)
integrated_test_df = dataprep.integrate_aux_data(test_df, processed_hdb_df)

In [ ]:
processed_train_df = dataprep.process_data(integrated_train_df)
processed_dev_df   = dataprep.process_data(integrated_dev_df)
processed_test_df  = dataprep.process_data(integrated_test_df)

In [ ]:
#processed_train_df.to_csv(intermediate_csv_path + 'processed_train.csv', index=False)
#processed_dev_df.to_csv(intermediate_csv_path + 'processed_val.csv', index=False)
#processed_test_df.to_csv(intermediate_csv_path + 'processed_test.csv', index=False)

# Section II-E: Feature Engineering and Post-processing

In [ ]:
added_train_df = dataprep.add_new_features(processed_train_df, is_test=False)
added_dev_df = dataprep.add_new_features(processed_dev_df, is_test=False)
added_test_df = dataprep.add_new_features(processed_test_df, is_test=True)

In [ ]:
#added_train_df.to_csv(intermediate_csv_path + 'added_train.csv', index=False)
#added_dev_df.to_csv(intermediate_csv_path + 'added_val.csv', index=False)
#added_test_df.to_csv(intermediate_csv_path + 'added_test.csv', index=False)

In [ ]:
# Generate original train, dev, test sets
orig_train_df, model_means, typemodel_means, town_means, subzone_means, planning_area_means, region_means = \
    dataprep.finalise_data(
        added_train_df,
        is_train=True,
        is_test=False  # choose 'sqm', 'rpi', 'both', or None depending on your preference
    )

orig_dev_df = dataprep.finalise_data(
    added_dev_df,
    is_train=False,
    is_test=False,
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

orig_test_df = dataprep.finalise_data(
    added_test_df,
    is_train=False,
    is_test=True,
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

In [ ]:
# Generate SQM-adjusted train, dev, test sets
sqm_train_df, model_means, typemodel_means, town_means, subzone_means, planning_area_means, region_means = \
    dataprep.finalise_data(
        added_train_df,
        is_train=True,
        is_test=False,
        adj='sqm'  # choose 'sqm', 'rpi', 'both', or None depending on your preference
    )

sqm_dev_df = dataprep.finalise_data(
    added_dev_df,
    is_train=False,
    is_test=False,
    adj='sqm',
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

sqm_test_df = dataprep.finalise_data(
    added_test_df,
    is_train=False,
    is_test=True,
    adj='sqm',
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

In [ ]:
# Generate RPI-adjusted train, dev, test sets
rpi_train_df, model_means, typemodel_means, town_means, subzone_means, planning_area_means, region_means = \
    dataprep.finalise_data(
        added_train_df,
        is_train=True,
        is_test=False,
        adj='rpi'  # choose 'sqm', 'rpi', 'both', or None depending on your preference
    )

rpi_dev_df = dataprep.finalise_data(
    added_dev_df,
    is_train=False,
    is_test=False,
    adj='rpi',
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

rpi_test_df = dataprep.finalise_data(
    added_test_df,
    is_train=False,
    is_test=True,
    adj='rpi',
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

In [ ]:
# Generate RPI-adjusted train, dev, test sets
sqmrpi_train_df, model_means, typemodel_means, town_means, subzone_means, planning_area_means, region_means = \
    dataprep.finalise_data(
        added_train_df,
        is_train=True,
        is_test=False,
        adj='both'  # choose 'sqm', 'rpi', 'both', or None depending on your preference
    )

sqmrpi_dev_df = dataprep.finalise_data(
    added_dev_df,
    is_train=False,
    is_test=False,
    adj='both',
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

sqmrpi_test_df = dataprep.finalise_data(
    added_test_df,
    is_train=False,
    is_test=True,
    adj='both',
    model_means=model_means,
    typemodel_means=typemodel_means,
    town_means=town_means,
    subzone_means=subzone_means,
    planning_area_means=planning_area_means,
    region_means=region_means
)

In [ ]:
dataprep.df_basic_info(orig_train_df, table_name="Original Train Set")
dataprep.df_basic_info(orig_dev_df, table_name="Original Dev Set")
dataprep.df_basic_info(orig_test_df, table_name="Original Test Set")


=== Original Train Set summary ===
╒═══════════════════════════╤════════╕
│ Number of records         │ 129971 │
├───────────────────────────┼────────┤
│ Number of features        │     56 │
├───────────────────────────┼────────┤
│ Number of NaN rows        │      0 │
├───────────────────────────┼────────┤
│ Number of duplicated rows │      0 │
╘═══════════════════════════╧════════╛

=== Original Dev Set summary ===
╒═══════════════════════════╤═══════╕
│ Number of records         │ 32493 │
├───────────────────────────┼───────┤
│ Number of features        │    56 │
├───────────────────────────┼───────┤
│ Number of NaN rows        │     0 │
├───────────────────────────┼───────┤
│ Number of duplicated rows │     0 │
╘═══════════════════════════╧═══════╛

=== Original Test Set summary ===
╒═══════════════════════════╤═══════╕
│ Number of records         │ 50000 │
├───────────────────────────┼───────┤
│ Number of features        │    55 │
├───────────────────────────┼───────┤
│ Number of 

In [ ]:
#dataprep.df_basic_info(sqm_train_df, table_name="SQM Train Set")
#dataprep.df_basic_info(sqm_dev_df, table_name="SQM Dev Set")
#dataprep.df_basic_info(sqm_test_df, table_name="SQM Test Set")

In [ ]:
#dataprep.df_basic_info(rpi_train_df, table_name="RPI Train Set")
#dataprep.df_basic_info(rpi_dev_df, table_name="RPI Dev Set")
#dataprep.df_basic_info(rpi_test_df, table_name="RPI Test Set")

In [ ]:
#dataprep.df_basic_info(sqmrpi_train_df, table_name="SQM+RPI Train Set")
#dataprep.df_basic_info(sqmrpi_dev_df, table_name="SQM+RPI Dev Set")
#dataprep.df_basic_info(sqmrpi_test_df, table_name="SQM_RPI Test Set")

In [ ]:
#orig_train_df.to_csv(intermediate_csv_path+'orig_train_df.csv', index=False)
#orig_dev_df.to_csv(intermediate_csv_path+'orig_dev_df.csv', index=False)
#orig_test_df.to_csv(intermediate_csv_path+'orig_test_df.csv', index=False)

In [ ]:
#sqm_train_df.to_csv(intermediate_csv_path+'sqm_train_df.csv', index=False)
#sqm_dev_df.to_csv(intermediate_csv_path+'sqm_dev_df.csv', index=False)
#sqm_test_df.to_csv(intermediate_csv_path+'sqm_test_df.csv', index=False)

In [ ]:
#rpi_train_df.to_csv(intermediate_csv_path+'rpi_train_df.csv', index=False)
#rpi_dev_df.to_csv(intermediate_csv_path+'rpi_dev_df.csv', index=False)
#rpi_test_df.to_csv(intermediate_csv_path+'rpi_test_df.csv', index=False)

In [ ]:
#sqmrpi_train_df.to_csv(intermediate_csv_path+'sqmrpi_train_df.csv', index=False)
#sqmrpi_dev_df.to_csv(intermediate_csv_path+'sqmrpi_dev_df.csv', index=False)
#sqmrpi_test_df.to_csv(intermediate_csv_path+'sqmrpi_test_df.csv', index=False)